<a href="https://colab.research.google.com/github/aymanr8-dot/DS-2002-Midterm-Project-Data-Demons-/blob/main/notebooks/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [1]:
team_setup = {
    'team_name': 'Data Demons',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Ayman Haque', 'github': 'aymanr8-dot'},
        {'name': 'Esther Man', 'github': 'EstherMan05'},
    ],
    'repo_url': 'https://github.com/aymanr8-dot/DS-2002-Midterm-Project-Data-Demons-',
    'charter_url': 'https://github.com/aymanr8-dot/DS-2002-Midterm-Project-Data-Demons-/blob/main/CHARTER.md',  # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Esther Man',
    'meeting_time': 'Tuesdays 4:30pm and Sundays 3:00pm, on Zoom',
    'five_worst_problems': [  # from Monday's profile of walmart_transactions.csv / store_locations.csv
        "store_id has 60 spellings for 12 stores ('FL-330', 'FL_330', 'FL330', 'fl-330', ' FL-330' with a leading space), "
        "so 4,345 rows do not join to the store file as-is; store_locations.csv also repeats Orlando and Kissimmee "
        "as 'FL239' and 'FL_287' and has blank latitude/longitude for Tallahassee (FL-619)",
        "unit_price is stored as text and 3,029 values start with '$', so it cannot be summed until stripped and cast to float",
        "timestamp comes in 5 formats ('2024-09-13 20:30:20', '2024-09-24T09:48:13', '09/26/2024 09:26', "
        "'09-23-2024 18:24:47', '21-Sep-2024 19:46'), some with no seconds, and 886 rows have no timestamp at all",
        "the same product sits under several SKUs: 45 SKUs for 20 real products (Pop-Tarts Strawberry alone is "
        "PT-12, POPTART-STRAW, PT_STRAWBERRY and SKU#459812), and category has 37 spellings for 10 categories",
        "130 rows have a negative quantity (-1 to -5) with no return flag, and 30 rows are exact duplicate transactions",
    ],
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [2]:
import shutil
import subprocess

# Colab has no copy of the repo on disk, so clone it here and point REPO_PATH at that clone.
REPO_URL = 'https://github.com/aymanr8-dot/DS-2002-Midterm-Project-Data-Demons-.git'
REPO_PATH = '/content/team-repo'   # your local clone of the team repo
shutil.rmtree(REPO_PATH, ignore_errors=True)
subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_PATH], check=True)

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

2ac0799 Add team charter (CHARTER.md at repo root)


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [3]:
import re
import numpy as np
import pandas as pd

DATA_URL = ('https://raw.githubusercontent.com/aymanr8-dot/DS-2002-Midterm-Project-Data-Demons-/'
            'main/notebooks/06-midterm-walmart/data/')

raw = pd.read_csv(DATA_URL + 'walmart_transactions.csv', dtype=str)
stores_raw = pd.read_csv(DATA_URL + 'store_locations.csv', dtype=str)

decision_log = []

def log_decision(step, decision, rows_touched, rows_after):
    decision_log.append({'step': step, 'decision': decision,
                         'rows_touched': int(rows_touched), 'rows_after': int(rows_after)})

def drop_duplicates(df):
    dupes = df.duplicated()
    out = df[~dupes].copy()
    log_decision('duplicates', 'Dropped exact duplicate rows (same transaction_id and every field)',
                 dupes.sum(), len(out))
    return out

def clean_store_id(s):
    """' FL-330', 'FL_330', 'FL330', 'fl-330' -> 'FL-330'"""
    s = s.astype(str).str.strip().str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
    return s.str.replace(r'^([A-Z]{2})(\d+)$', r'\1-\2', regex=True)

def standardize_store_ids(df):
    before = df['store_id']
    df['store_id'] = clean_store_id(before)
    log_decision('store_id', f"Stripped spaces, upper-cased, rewrote to FL-### "
                 f"({before.nunique()} spellings -> {df['store_id'].nunique()} stores)",
                 (before != df['store_id']).sum(), len(df))
    return df

def parse_unit_price(df):
    had_dollar = df['unit_price'].str.contains(r'\$', na=False)
    df['unit_price'] = pd.to_numeric(df['unit_price'].str.replace(r'[$,\s]', '', regex=True),
                                     errors='coerce')
    assert df['unit_price'].notna().all(), 'some unit_price values did not convert'
    log_decision('unit_price', "Removed leading '$' and converted text to float",
                 had_dollar.sum(), len(df))
    return df

def flag_returns(df):
    df['quantity'] = pd.to_numeric(df['quantity']).astype(int)
    df['is_return'] = df['quantity'] < 0
    log_decision('quantity', 'Negative quantities treated as returns: kept, flagged is_return=True, '
                 'excluded from demand counts', df['is_return'].sum(), len(df))
    return df

TS_FORMATS = ['%Y-%m-%d %H:%M:%S',   # 2024-09-13 20:30:20
              '%Y-%m-%dT%H:%M:%S',   # 2024-09-24T09:48:13
              '%m/%d/%Y %H:%M',      # 09/26/2024 09:26
              '%m-%d-%Y %H:%M:%S',   # 09-23-2024 18:24:47
              '%d-%b-%Y %H:%M']      # 21-Sep-2024 19:46

def parse_timestamps(df):
    s = df['timestamp'].str.strip()
    parsed = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
    for fmt in TS_FORMATS:
        parsed = parsed.fillna(pd.to_datetime(s, format=fmt, errors='coerce'))
    unparsed = s.notna() & parsed.isna()
    assert unparsed.sum() == 0, f'{unparsed.sum()} timestamps in an unknown format'
    reformatted = s.notna() & ~s.str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$', na=False)
    df['timestamp'] = parsed
    log_decision('timestamp', f'Parsed {len(TS_FORMATS)} text formats to datetime (store-local time)',
                 reformatted.sum(), len(df))
    df['timestamp_missing'] = df['timestamp'].isna()
    log_decision('timestamp', 'Missing timestamps flagged (timestamp_missing=True), not dropped: still count '
                 'for SKU/product totals, left out of anything time-based', df['timestamp_missing'].sum(), len(df))
    return df

CATEGORY_MAP = {'emergency': 'Emergency', 'emergency supplies': 'Emergency',
                'grocery': 'Grocery', 'groceries': 'Grocery',
                'electronics': 'Electronics',
                'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
                'household': 'Household', 'household essentials': 'Household',
                'baby': 'Baby', 'baby products': 'Baby',
                'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
                'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
                'snacks': 'Snacks',
                'health': 'Health', 'health & wellness': 'Health'}

def standardize_categories(df):
    before = df['category']
    df['category'] = before.str.strip().str.lower().map(CATEGORY_MAP)
    assert df['category'].notna().all(), 'unmapped category label'
    log_decision('category', f"Mapped {before.nunique()} category spellings to {df['category'].nunique()} categories",
                 (before != df['category']).sum(), len(df))
    df['product_name'] = df['product_name'].str.strip()
    return df

STORE_TZ = {'FL-710': 'America/Chicago'}   # Pensacola is on Central time; every other store is Eastern

def clean_stores(stores):
    s = stores.copy()
    s['store_id'] = clean_store_id(s['store_id'])
    s = s.drop_duplicates('store_id').copy()            # FL239 / FL_287 were repeat rows
    s['state'] = 'FL'                                    # 'Florida', 'Fla.' -> 'FL'
    s['region'] = s['region'].str.strip()
    s[['latitude', 'longitude']] = s[['latitude', 'longitude']].astype(float)
    tlh = s['store_id'] == 'FL-619'                      # Tallahassee had blank coordinates (filled with city center)
    s.loc[tlh, ['latitude', 'longitude']] = [30.4383, -84.2807]
    s['tz'] = s['store_id'].map(STORE_TZ).fillna('America/New_York')
    return s.reset_index(drop=True)

def clean_transactions(df):
    df = drop_duplicates(df)
    df = standardize_store_ids(df)
    df = parse_unit_price(df)
    df = flag_returns(df)
    df = parse_timestamps(df)
    df = standardize_categories(df)
    return df.reset_index(drop=True)

sales = clean_transactions(raw)
stores = clean_stores(stores_raw)

print(f'rows before cleaning: {len(raw):,}')
print(f'rows after cleaning:  {len(sales):,}')
print(f'rows removed:         {len(raw) - len(sales):,}')
print(f"returns flagged (kept): {sales['is_return'].sum():,}")
print(f"missing timestamps flagged (kept): {sales['timestamp_missing'].sum():,}")
print(f"store file: {len(stores_raw)} rows -> {len(stores)} stores after dedupe")
unjoined = set(sales['store_id']) - set(stores['store_id'])
print('store_ids that do not join to the store file:', unjoined or 'none')
print('date range:', sales['timestamp'].min(), 'to', sales['timestamp'].max())
sales.head()

rows before cleaning: 25,562
rows after cleaning:  25,532
rows removed:         30
returns flagged (kept): 130
missing timestamps flagged (kept): 884
store file: 14 rows -> 12 stores after dedupe
store_ids that do not join to the store file: none
date range: 2024-09-01 06:01:15 to 2024-09-30 22:57:47


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,is_return,timestamp_missing
0,TXN012636,FL-710,2024-09-24 09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15,False,False
1,TXN000759,FL-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,Grocery,1,2.02,False,False
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household,2,10.47,False,False
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,Electronics,1,28.59,False,False
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcohol,3,18.82,False,False


### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [4]:
log_df = pd.DataFrame(decision_log)
pd.set_option('display.max_colwidth', None)
log_df

,step,decision,rows_touched,rows_after
0,duplicates,Dropped exact duplicate rows (same transaction_id and every field),30,25532
1,store_id,"Stripped spaces, upper-cased, rewrote to FL-### (60 spellings -> 12 stores)",4345,25532
2,unit_price,Removed leading '$' and converted text to float,3026,25532
3,quantity,"Negative quantities treated as returns: kept, flagged is_return=True, excluded from demand counts",130,25532
4,timestamp,Parsed 5 text formats to datetime (store-local time),12479,25532
5,timestamp,"Missing timestamps flagged (timestamp_missing=True), not dropped: still count for SKU/product totals, left out of anything time-based",884,25532
6,category,Mapped 37 category spellings to 10 categories,18744,25532


### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [5]:
# Canonical product = keyword rule on the normalized product name.
# Each SKU gets the canonical product its transaction names map to, then we check
# (a) every name under a SKU agrees and (b) prices inside each group sit in one band.
# Decision: aggregate across package sizes. The catalog's package_size contradicts the names
# (e.g. 'BOTTLED WATER 24PK' listed as '1 pack') and every SKU in a group sells in the same
# ~10% price band, so these are one product under different codes, not different pack sizes.
# Monopoly and Playing Cards are kept as separate products on purpose (case brief, Q5).
PRODUCT_RULES = [
    ('Pop-Tarts Strawberry',   r'pop ?tart'),
    ('Bottled Water 24-Pack',  r'water'),
    ('AA Batteries 4-Pack',    r'batter'),
    ('LED Flashlight',         r'flashlight'),
    ('Light Beer 24-Pack',     r'beer'),
    ('Monopoly Board Game',    r'monopoly'),
    ('Playing Cards',          r'playing cards'),
    ('Roku Streaming Stick',   r'roku|streaming'),
    ('Canned Soup',            r'soup'),
    ('White Bread',            r'bread'),
    ('Peanut Butter',          r'peanut'),
    ('Emergency Candles',      r'candle'),
    ('First Aid Kit',          r'first ?aid'),
    ('Portable Phone Charger', r'charger'),
    ('Chips Variety Pack',     r'chips'),
    ('Paper Towels',           r'paper towel'),
    ('Toilet Paper 12-Roll',   r'toilet|\btp\b'),
    ('Baby Diapers Size 3',    r'diaper'),
    ('Baby Formula',           r'formula'),
    ('Heavy Duty Tarp',        r'tarp'),
]

def normalize_name(name):
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9]+', ' ', str(name).lower())).strip()

def canonical_for(name):
    n = normalize_name(name)
    hits = [prod for prod, pat in PRODUCT_RULES if re.search(pat, n)]
    return hits[0] if len(hits) == 1 else None      # None = no match or ambiguous

def build_sku_map(df):
    names = df[['sku', 'product_name']].drop_duplicates().copy()
    names['canonical'] = names['product_name'].map(canonical_for)
    unmatched = names[names['canonical'].isna()]
    assert unmatched.empty, f'names with no single rule match:\n{unmatched}'
    per_sku = names.groupby('sku')['canonical'].agg(lambda s: sorted(set(s)))
    conflicts = per_sku[per_sku.str.len() > 1]
    assert conflicts.empty, f'SKUs whose names disagree:\n{conflicts}'
    return per_sku.str[0].rename('canonical_product')

catalog = pd.read_csv(DATA_URL + 'product_catalog.csv', dtype=str)
sku_map = build_sku_map(sales)
sales['canonical_product'] = sales['sku'].map(sku_map)

groups = (sales.groupby('canonical_product')
               .agg(skus=('sku', 'nunique'), rows=('sku', 'size'),
                    min_price=('unit_price', 'min'), max_price=('unit_price', 'max'))
               .sort_values(['skus', 'rows'], ascending=False))
groups['price_spread'] = (groups['max_price'] / groups['min_price']).round(2)

largest = groups.index[0]
print(f"distinct SKUs at start:   {sales['sku'].nunique()}")
print(f"canonical products now:   {sales['canonical_product'].nunique()}")
print(f"largest group:            {largest} ({groups.loc[largest, 'skus']} SKUs, "
      f"{groups.loc[largest, 'rows']:,} rows)")
print(f"max price spread in any group: {groups['price_spread'].max()}x (all SKUs in a group share one price band)")

print('\nPop-Tarts family mapping:')
poptarts = (sales[sales['canonical_product'] == 'Pop-Tarts Strawberry']
            .groupby('sku')
            .agg(rows=('sku', 'size'), median_price=('unit_price', 'median'),
                 name_spellings=('product_name', 'nunique'))
            .join(catalog.drop_duplicates('sku').set_index('sku')[['product_name', 'supplier', 'package_size']]
                  .rename(columns={'product_name': 'catalog_name'})))
display(poptarts)
display(groups)

distinct SKUs at start:   45
canonical products now:   20
largest group:            Pop-Tarts Strawberry (4 SKUs, 2,062 rows)
max price spread in any group: 1.11x (all SKUs in a group share one price band)

Pop-Tarts family mapping:


,rows,median_price,name_spellings,catalog_name,supplier,package_size
sku,,,,,,
POPTART-STRAW,506,3.49,5,Pop-Tarts Strawberry,Sara Lee,6 pack
PT-12,538,3.49,5,pop-tarts strawberry,Johnson & Johnson,4-pack
PT_STRAWBERRY,515,3.50,5,pop-tarts strawberry,Campbell's,4 pack
SKU#459812,503,3.48,5,Strawberry Pop-Tart,Roku Inc,4pk


,skus,rows,min_price,max_price,price_spread
canonical_product,,,,,
Pop-Tarts Strawberry,4,2062,3.32,3.66,1.10
Bottled Water 24-Pack,3,1489,4.74,5.24,1.11
AA Batteries 4-Pack,3,1462,6.17,6.81,1.10
Monopoly Board Game,3,1339,18.99,20.98,1.10
Heavy Duty Tarp,2,1466,18.99,20.99,1.11
LED Flashlight,2,1460,12.34,13.64,1.11
Emergency Candles,2,1313,5.69,6.29,1.11
Portable Phone Charger,2,1313,23.75,26.24,1.10
Light Beer 24-Pack,2,1265,18.04,19.94,1.11


### 4 — One working weather API call — not started

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [6]:
# not started

### 5 — One chart — not started

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [7]:
# not started

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [8]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),  # from the team setup section
    'charter_committed': bool(result.stdout.strip()),  # True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': len(sales),                               # an integer
    'decisions_logged': len(decision_log),                           # how many entries in your log
    'canonical_products': int(sales['canonical_product'].nunique()), # distinct products after consolidation
    'weather_api_rows': None,                                        # rows returned by your API call (section 4 not started)
    'notebook_runs_clean': True,                                     # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', 5)
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  done - rows_after_cleaning = 25532
  done - decisions_logged = 7
  done - canonical_products = 20
  not yet - weather_api_rows = None
  done - notebook_runs_clean = True

required items complete: True
progress items done: 4 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** Both of us have a lot of scheduling conflicts over the next week, so we will have fewer chances to sit down and work together. That is most likely to slow down Q3 (weather correlation) and Q4 (stores and geography), which both still need real analysis. If they slip, everything lands in the final weekend and we end up cramming the analysis and the presentation (Mon Nov 2) at the same time, with no time to rehearse.

**Plan, with an owner and a date:** we will stop depending on finding a shared time slot and work independently, while keeping each other up to date throughout the week.
- **Both, starting Mon Oct 26:** each of us works in our own notebook under `notebooks/scratch/`, pulls before starting and pushes before stopping, and hands finished code to the week's notebook owner as functions. This keeps us from overwriting each other.
- **Both, every day we work on the project:** post a short update in our group chat covering what changed, which numbers moved, and what is blocked. Any change to the cleaning goes in the decision log the same day, so neither of us builds on an out-of-date version.
- **Ayman:** Q3 correlation and lag analysis pushed by Thu Oct 29, and Q4 store comparison pushed by Sat Oct 31.
- **Esther:** Q1 and Q2 pushed by Thu Oct 29, so both are in the Lab 10 notebook on Fri Oct 30.
- **Both:** build presentation slides as each question is finished rather than at the end, and use our Sun Nov 1 3pm meeting to rehearse once.

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | in progress: timestamps cleaned and parsed (section 1); baseline window and daily chart not built yet | Esther |
| Q2 SKU consolidation | in progress: 45 SKUs mapped to 20 products (section 3); before/after comparison not built yet | Esther |
| Q3 weather correlation | not started: the weather API call (section 4) is not built yet | Ayman |
| Q4 store and geography | not started | Ayman |
| Q5 signal or artifact | not started | Esther & Ayman |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.